## Figure 3: direct loss parameter $\xi_h$

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D # Needed for custom legend handles
import os # Import os for os.getcwd()

# Load the CSV file
try:
    df_inputs = pd.read_csv('output/xi_h_inputs_mean.csv') # Corrected filename
    print("File 'output/xi_h_inputs_mean.csv' loaded successfully.")
except FileNotFoundError:
    print("Error: 'output/xi_h_inputs_mean.csv' not found. Please ensure the file is in the current working directory or provide the correct path.")
    print(f"Current working directory: {os.getcwd()}")
    # Create a dummy DataFrame for demonstration if the file is not found
    data = {
        'county': ['County A', 'County A', 'County B', 'County B', 'County A', 'County A', 'County B', 'County B'], # Corrected column name
        'return_period': [10, 100, 10, 100, 10, 100, 10, 100],
        'climate_scenario': ['Baseline', 'Baseline', 'Baseline', 'Baseline', 'Climate', 'Climate', 'Climate', 'Climate'], # Corrected column name and values
        'mangrove_state': ['No_Mangroves', 'No_Mangroves', 'No_Mangroves', 'No_Mangroves', 'With_Mangroves', 'With_Mangroves', 'With_Mangroves', 'With_Mangroves'], # Corrected column name and values
        'xi_h': [0.05, 0.08, 0.06, 0.09, 0.02, 0.03, 0.025, 0.035] # Corrected column name
    }
    df_inputs = pd.DataFrame(data)
    print("Using a dummy DataFrame for demonstration.")

# Define custom colors for mangrove status, matching the values in 'mangrove_state'
mangrove_colors = {
    'No_Mangroves': '#A52A2A',  # Brick red base
    'With_Mangroves': '#228B22' # Forest green base
}

# Define custom markers for climate scenario, matching the values in 'climate_scenario'
scenario_markers = {
    'Baseline': 'o',  # Circle
    'Climate': 'X'    # 'X' marker
}

plt.figure(figsize=(12, 7))

# Get unique return periods to map colors correctly
unique_rps = sorted(df_inputs['return_period'].unique())
num_shades = len(unique_rps)

# Define palettes for each mangrove state using Seaborn's built-in color palettes
palettes = {
    'No_Mangroves': sns.color_palette("Reds", n_colors=num_shades), # Red gamma
    'With_Mangroves': sns.color_palette("Greens", n_colors=num_shades) # Green gamma
}

# --- Prepare data for offset ---
# Get unique counties and their order for x-axis positioning
ordered_counties = df_inputs['county'].unique()
county_to_numeric = {county: i for i, county in enumerate(ordered_counties)}

# Create a temporary DataFrame for plotting to add offset positions
df_inputs_plot = df_inputs.copy()
df_inputs_plot['numeric_county'] = df_inputs_plot['county'].map(county_to_numeric)

# Define offset amount for marker separation for climate scenario
scenario_offset_amount = 0.15 # Adjust as needed for visual separation

# Define additional offset for mangrove state
mangrove_state_offset_amount = 0.03 # Smaller offset to fine-tune separation

# Apply offset based on scenario type AND mangrove state
df_inputs_plot['x_offset_position'] = df_inputs_plot.apply(
    lambda row:
        row['numeric_county']
        + (-scenario_offset_amount if row['climate_scenario'] == 'Baseline' else scenario_offset_amount)
        + (-mangrove_state_offset_amount if row['mangrove_state'] == 'No_Mangroves' else mangrove_state_offset_amount),
    axis=1
)

# List to hold all legend elements
legend_elements = []

# First, plot all scatter points without legend creation
for status, base_color in mangrove_colors.items():
    df_status_plot = df_inputs_plot[df_inputs_plot['mangrove_state'] == status]
    if not df_status_plot.empty:
        # Create a dictionary to map return periods to specific colors within the current gamma
        rp_to_color_map = {rp: palettes[status][i] for i, rp in enumerate(unique_rps)}

        sns.scatterplot(
            data=df_status_plot,
            x='xi_h', # Swapped x and y axes
            y='x_offset_position', # Swapped x and y axes
            hue='return_period',
            s=200,
            style='climate_scenario',
            markers=scenario_markers,
            palette=rp_to_color_map,
            alpha=0.8,
            edgecolor=base_color, # Add a solid outline matching the base color
            linewidth=1.5,      # Set linewidth for visibility
            legend=False # Do not generate legend automatically here
        )

# Manually create legend elements for mangrove status (No mangroves / With mangroves)
# Using a simple circle marker with the base color
legend_elements.append(Line2D([0], [0], marker='o', color='w', label='No mangroves', markerfacecolor='#A52A2A', markersize=10))
legend_elements.append(Line2D([0], [0], marker='o', color='w', label='With mangroves', markerfacecolor='#228B22', markersize=10))

# Now, manually create legend elements for return_period (both red and green shades) and climate_scenario
# Add legend elements for return periods for 'No_Mangroves' (red shades)
rp_legend_colors_red = {rp: palettes['No_Mangroves'][i] for i, rp in enumerate(unique_rps)}
for rp in unique_rps:
    legend_elements.append(Line2D([0], [0], marker='o', color='w', label=f'RP {rp} (No mangroves)',
                                  markerfacecolor=rp_legend_colors_red[rp], markersize=10))

# Add legend elements for return periods for 'With_Mangroves' (green shades)
rp_legend_colors_green = {rp: palettes['With_Mangroves'][i] for i, rp in enumerate(unique_rps)}
for rp in unique_rps:
    legend_elements.append(Line2D([0], [0], marker='o', color='w', label=f'RP {rp} (With mangroves)',
                                  markerfacecolor=rp_legend_colors_green[rp], markersize=10))

# Add legend elements for climate scenarios
for scenario_name, marker_style in scenario_markers.items():
    legend_elements.append(Line2D([0], [0], marker=marker_style, color='black', label=f'Scenario: {scenario_name}',
                                  markerfacecolor='black', markersize=10, linestyle='None')) # linestyle='None' for marker only

# Add text labels above the horizontal lines at Charlotte county marker level
# Find the index of 'Charlotte' county from the ordered list
try:
    charlotte_idx = list(ordered_counties).index('Charlotte')
except ValueError:
    charlotte_idx = 0 # Default to first county if 'Charlotte' not found

# Adjust text position slightly above the line
text_offset_y = 0.005

plt.title(' ', fontsize=18)
plt.ylabel(' ', fontsize=18) # Keep label as County Name for readability (now on y-axis)
plt.xlabel('Share of structure value lost due to storm: 'r'$\xi_h$', fontsize=18) # Update label based on the actual y-axis data column (now on x-axis)

# Set custom y-axis ticks to match the numeric positions and label them with county names
plt.yticks(ticks=list(county_to_numeric.values()), labels=ordered_counties, rotation=0, ha='right', fontsize=18)
plt.xticks(fontsize=18)
plt.grid(True, linestyle=':', alpha=0.6)

# Create the final legend using the collected elements
plt.legend(handles=legend_elements, bbox_to_anchor=(1.02, 1), loc='upper left', title='Legend', fontsize=16) # Increased fontsize to 16 and pushed slightly right
plt.tight_layout()

# Save the figure as a PNG file
plt.savefig('output/xiPA', dpi=300, bbox_inches='tight')

plt.show()

## Figure 4: mangrove protection value

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np # Import numpy for numerical operations like range
import matplotlib.ticker # Import for tick formatting
from matplotlib.lines import Line2D # Import Line2D for custom legend handles
from matplotlib.patches import Patch, PathPatch # Import Patch for custom legend handles for bar charts
from matplotlib.path import Path

# Font size settings
title_fontsize = 22 # Increased
label_fontsize = 20 # Increased
tick_fontsize = 18 # Increased
legend_fontsize = 12 # Not increased, as per request

# Helper function to draw a broken axis indicator (two diagonal lines)
def draw_broken_axis_indicator(ax, y_center, bar_height, cut_position):
    lw = 1.5 # Line width for the cut

    # Horizontal span of the cut indicator
    cut_length = 0.015 * ax.get_xlim()[1] # Relative length of the cut mark

    # Vertical span of the cut indicator relative to bar height
    cut_vertical_span = bar_height * 0.7 # Occupy 70% of bar height

    # Coordinates for the two diagonal lines making the cut
    # Line 1
    ax.plot([cut_position - cut_length/2, cut_position + cut_length/2],
            [y_center - cut_vertical_span/2, y_center + cut_vertical_span/2],
            color='white', lw=lw + 2, solid_capstyle='round', clip_on=False, zorder=4) # White background for visibility
    ax.plot([cut_position - cut_length/2, cut_position + cut_length/2],
            [y_center - cut_vertical_span/2, y_center + cut_vertical_span/2],
            color='black', lw=lw, solid_capstyle='round', clip_on=False, zorder=5)

    # Line 2 (slightly offset for "wiggly" feel)
    ax.plot([cut_position - cut_length/2 + cut_length/4, cut_position + cut_length/2 + cut_length/4],
            [y_center + cut_vertical_span/2, y_center - cut_vertical_span/2],
            color='white', lw=lw + 2, solid_capstyle='round', clip_on=False, zorder=4)
    ax.plot([cut_position - cut_length/2 + cut_length/4, cut_position + cut_length/2 + cut_length/4],
            [y_center + cut_vertical_span/2, y_center - cut_vertical_span/2],
            color='black', lw=lw, solid_capstyle='round', clip_on=False, zorder=5)

# Define the exact columns to plot from the user's request
value_vars_to_plot = [
    'protection_pct',
    'per_property_protection',
    'aggregate_at_risk_protection',
    'normalized_protection_pct'
]

# Mapping for new titles (A, B, C, D)
title_map = {
    'protection_pct': 'A. Share of property value',
    'per_property_protection': 'B. Per property (mean)',
    'aggregate_at_risk_protection': 'C. County aggregate',
    'normalized_protection_pct': 'D. Share of market value'
}

df_results = None # Initialize df_results to None

# Load the CSV file
try:
    df_results = pd.read_csv('output/assessment_results_long.csv') # Changed from Results.csv to assessment_results_long.csv
    print("File 'output/assessment_results_long.csv' loaded successfully.")
except FileNotFoundError:
    print("Error: 'output/assessment_results_long.csv' not found. Please ensure the file is in the current working directory.")

# Only proceed if df_results was loaded
if df_results is not None:
    # Check if the required columns exist in df_results.
    missing_cols = [col for col in value_vars_to_plot if col not in df_results.columns]
    if missing_cols:
        print(f"Error: Loaded 'output/assessment_results_long.csv' is missing required columns: {', '.join(missing_cols)}. Cannot generate plot.")
        df_results = None # Set df_results to None to prevent plotting
    else:
        # Melt the DataFrame to prepare for faceting by variable
        df_melted = df_results.melt(
            id_vars=['county', 'scenario'],
            value_vars=value_vars_to_plot, # Use the user-specified value_vars
            var_name='variable',
            value_name='value'
        )

        # Define custom colors for scenarios
        scenario_colors = {
            'baseline': 'royalblue',
            'climate_change': 'darkred',
            'market': 'orange'
        }

        # Create the FacetGrid
        if not df_melted.empty:
            g = sns.catplot(
                data=df_melted,
                x='value', # Swapped x and y for horizontal bars
                y='county',
                hue='scenario',
                col='variable', # Create separate columns for each variable
                kind='bar',
                palette=scenario_colors,
                height=10, # Increased for taller plots
                aspect=0.42, # Decreased proportionally to maintain same width
                col_wrap=4, # Arrange plots in 4 columns for a single row
                sharex=False, # Allow x-axes to scale independently for different variables
                sharey=True, # Keep sharey=True as it's common for these types of plots
                legend=False # Set to False to disable automatic legend generation from catplot
            )

            # Adjust titles and labels for each subplot
            g.set_xticklabels(rotation=0, fontsize=tick_fontsize) # X-axis labels (values) no rotation
            g.set_ylabels('') # Remove generic y-axis title for all subplots

            # Get the unique county names to set as y-tick labels
            county_names = df_melted['county'].unique()

            # Set individual titles, x-axis labels, and adjust x-axis limits for each subplot
            for i, ax in enumerate(g.axes.flat):
                variable_name = df_melted['variable'].unique()[i]
                # Set new title based on title_map
                ax.set_title(title_map.get(variable_name, ''), fontsize=title_fontsize) # Use title map
                ax.grid(axis='x', linestyle=':', alpha=0.6)

                # Ensure y-axis labels (county names) are visible on the left-most chart
                if i == 0: # This is the first subplot (left-most)
                    ax.set_ylabel('County Name', fontsize=label_fontsize) # Add y-label title
                    # Set ticks first to prevent the UserWarning
                    ax.set_yticks(np.arange(len(county_names)))
                    ax.set_yticklabels(county_names, fontsize=tick_fontsize) # Explicitly set tick labels and their font size

                # Adjust x-axis limits and tick formatting for specific charts
                if variable_name in ['protection_pct', 'normalized_protection_pct']:
                    ax.set_xlim(left=0) # Ensure positive values start from 0
                    ax.set_xlabel('Percent', fontsize=label_fontsize)
                elif variable_name == 'per_property_protection':
                    x_max_display_per_prop = 60000 # Define truncation limit for this specific chart
                    ax.set_xlim(left=0, right=x_max_display_per_prop + 0.05 * x_max_display_per_prop) # Set x-limit and add space for labels
                    formatter = matplotlib.ticker.FuncFormatter(lambda x, p: f'{x/1000:.0f}')
                    ax.xaxis.set_major_formatter(formatter)
                    ax.xaxis.set_major_locator(matplotlib.ticker.MaxNLocator(nbins=6)) # Reduce number of ticks
                    ax.set_xlabel('Thousands USD', fontsize=label_fontsize)
                elif variable_name == 'aggregate_at_risk_protection':
                    ax.set_xlim(left=0)
                    formatter = matplotlib.ticker.FuncFormatter(lambda x, p: f'{x/1000000000:.1f}')
                    ax.xaxis.set_major_formatter(formatter)
                    ax.set_xlabel('Billions USD', fontsize=label_fontsize)

                # Get the ordered list of unique scenarios from df_melted for consistent mapping
                ordered_scenarios = sorted(df_melted['scenario'].unique())

                # Add data labels to bars
                for container_idx, container in enumerate(ax.containers):
                    # Get the actual scenario name corresponding to this container
                    scenario_label = ordered_scenarios[container_idx]

                    # Retrieve original data for this scenario and variable
                    df_current_bars = df_melted[
                        (df_melted['variable'] == variable_name) &
                        (df_melted['scenario'] == scenario_label)
                    ].sort_values(by='county') # Ensure order matches plotting order for counties

                    for bar_idx, bar in enumerate(container):
                        # Safely access the original_value
                        if bar_idx >= len(df_current_bars):
                            continue

                        original_value = df_current_bars.iloc[bar_idx]['value']

                        # Define format string based on variable
                        if variable_name in ['protection_pct', 'normalized_protection_pct']:
                            label_text = f'{original_value:.1f}%'
                        elif variable_name == 'per_property_protection':
                            label_text = f'{original_value/1000:.1f}K'
                        elif variable_name == 'aggregate_at_risk_protection':
                            label_text = f'{original_value/1000000000:.1f}B'
                        else:
                            label_text = f'{original_value:.1f}' # Default numeric format

                        # Truncation logic specific to 'per_property_protection'
                        if variable_name == 'per_property_protection':
                            x_max_display_per_prop = 60000

                            if original_value > x_max_display_per_prop:
                                # Truncate the bar visually
                                bar.set_width(x_max_display_per_prop)
                                # Draw broken axis indicator
                                draw_broken_axis_indicator(ax, bar.get_y() + bar.get_height()/2, bar.get_height(), x_max_display_per_prop)
                                # Place label for original value near the cut
                                ax.text(
                                    x_max_display_per_prop + 0.005 * ax.get_xlim()[1], # Position right of the cut
                                    bar.get_y() + bar.get_height() / 2, # Center vertically
                                    label_text, # Original value text
                                    ha='left', va='center', fontsize=14, color='black'
                                )
                                continue # Skip regular label placement for this truncated bar

                        # Regular label placement for non-truncated bars or other variables
                        # Use bar.get_width() for positioning, as it reflects the actual plotted bar length
                        if scenario_label == 'market': # For market scenario (orange bars)
                            ax.text(
                                bar.get_x() + bar.get_width() / 2, # Center horizontally
                                bar.get_y() + bar.get_height(), # Place above the bar
                                label_text,
                                ha='center', va='bottom', fontsize=14, color='black'
                            )
                        else: # For other scenarios (baseline, climate_change)
                            ax.text(
                                bar.get_x() + bar.get_width() + 0.005 * ax.get_xlim()[1], # Place at the right edge of the bar with relative offset
                                bar.get_y() + bar.get_height() / 2, # Center vertically
                                label_text,
                                ha='left', va='center', fontsize=14, color='black'
                            )

            plt.suptitle(' ', y=1.02, fontsize=title_fontsize+2)

            # Manually create legend elements using Patch for bar plots
            custom_legend_elements = []
            for scenario_name, color_val in scenario_colors.items():
                custom_legend_elements.append(Patch(facecolor=color_val, label=scenario_name.replace('_', ' ').title()))

            # Get the position of the last subplot (chart 4)
            last_ax_position = g.axes.flat[3].get_position()
            x_anchor = last_ax_position.x1 + 0.035 # Increased rightward offset
            y_anchor = last_ax_position.y0 + 0.05 # Move legend up by increasing y_anchor

            # Add the legend to the FacetGrid using custom handles and labels
            g.add_legend(handles=custom_legend_elements, title='Scenario', bbox_to_anchor=(x_anchor, y_anchor), loc='lower right', borderaxespad=0., fontsize=16)

            plt.tight_layout() # Adjust layout to prevent overlapping elements
            plt.savefig('output/allresults_w.png', dpi=300, bbox_inches='tight')
            plt.show()
        else:
            print("No data to plot after melting. Please check the 'output/assessment_results_long.csv' file contents.")
else:
    print("Plotting aborted due to missing file or columns.")

## Figure 2: storm timeline

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load the CSV file
try:
    df_storms_replicated = pd.read_csv('output/storm_timeline_data.csv')
    print("File 'output/storm_timeline_data.csv' loaded successfully.")
except FileNotFoundError:
    print("Error: 'output/storm_timeline_data.csv' not found. Please ensure the file is in the current working directory.")

years_replicated = df_storms_replicated['year'].unique()

storm_counts_baseline_replicated = np.zeros((len(years_replicated), 4))
storm_counts_climate_replicated = np.zeros((len(years_replicated), 4))

rp_labels = ['RP10', 'RP25', 'RP50', 'RP100']

for year_idx, year in enumerate(years_replicated):
    for storm_idx, rp_label in enumerate(rp_labels):
        baseline_row = df_storms_replicated[
            (df_storms_replicated['year'] == year) &
            (df_storms_replicated['scenario'] == 'baseline') &
            (df_storms_replicated['return_period'] == rp_label)
        ]
        if not baseline_row.empty:
            storm_counts_baseline_replicated[year_idx, storm_idx] = baseline_row['mean_num_storms'].iloc[0]

        climate_row = df_storms_replicated[
            (df_storms_replicated['year'] == year) &
            (df_storms_replicated['scenario'] == 'climate_change') &
            (df_storms_replicated['return_period'] == rp_label)
        ]
        if not climate_row.empty:
            storm_counts_climate_replicated[year_idx, storm_idx] = climate_row['mean_num_storms'].iloc[0]

fig_replicated, ax = plt.subplots(figsize=(16, 6.5))

colors = ['#3498db', '#e74c3c', '#f39c12', '#9b59b6']
labels = ['10-yr storms', '25-yr storms', '50-yr storms', '100-yr storms']
baseline_marker = 'o'
climate_marker = '^'

for storm_idx in range(4):
    current_color = colors[storm_idx]
    current_label = labels[storm_idx]

    baseline_data_y = storm_counts_baseline_replicated[:, storm_idx]
    climate_data_y = storm_counts_climate_replicated[:, storm_idx]

    ax.scatter(years_replicated, baseline_data_y,
               color=current_color,
               label=f'{current_label} (Baseline)',
               marker=baseline_marker,
               s=50, alpha=0.7, edgecolors='white', linewidth=0.5)

    ax.scatter(years_replicated, climate_data_y,
               color=current_color,
               label=f'{current_label} (Climate Change)',
               marker=climate_marker,
               s=70, alpha=0.7, edgecolors='white', linewidth=0.5)

    for year_idx, year in enumerate(years_replicated):
        ax.plot([year, year], [baseline_data_y[year_idx], climate_data_y[year_idx]],
                color=current_color, linestyle='-', linewidth=1, alpha=0.5)

ax.set_xlabel('Year', fontsize=18, fontweight='bold')
ax.set_ylabel('Frequency', fontsize=18, fontweight='bold')
ax.set_title(' ', fontsize=20, fontweight='bold', pad=15)

# LEGEND ADJUSTMENT: Moved bbox_to_anchor from -0.1 to -0.15 to push it lower
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), fontsize=18, framealpha=0.9, ncol=3)

ax.tick_params(axis='x', labelsize=18)
ax.tick_params(axis='y', labelsize=18)
ax.grid(True, alpha=0.3, linestyle='--')
ax.set_xlim(years_replicated.min() - 1, years_replicated.max() + 1)
ax.set_ylim(0, max(np.max(storm_counts_baseline_replicated), np.max(storm_counts_climate_replicated)) * 1.1)
ax.set_facecolor('#f8f9fa')

plt.tight_layout()
plt.savefig('output/storm_timeline_combined.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()